# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect the package description with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access metadata (do not subscript the metadata object)
metadata_obj = dataset.metadata
print(f"Dataset name: {metadata_obj.name}")
print(f"Description: {metadata_obj.description}\n")
# Optionally display other high-level metadata attributes
print("Published on:", getattr(metadata_obj, 'datePublished', 'N/A'))
print("Version:", getattr(metadata_obj, 'version', 'N/A'))
print("Cite as:", getattr(metadata_obj, 'citeAs', 'N/A'))

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s using the Croissant metadata structure.

In [ ]:
print("\n--- Record Sets in the dataset ---")
record_sets = metadata_obj.recordSet  # This returns a list of RecordSet objects
record_set_ids = []
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '[no name]')}")
    record_set_ids.append(rs['@id'])
    # List fields for each record set
    fields = rs.get('field', [])
    print("  Fields via @id:")
    for f in fields:
        try:
            # If full field object
            f_id = f["@id"]
            f_name = f.get('name', '[no name]')
        except Exception:
            f_id = str(f)
            f_name = '[unknown]'
        print(f"    - {f_id} ({f_name})")
    print("  Columns:")
    columns = rs.get('column', [])
    for c in columns:
        try:
            c_id = c["@id"]
            c_name = c.get('name', '[no name]')
        except Exception:
            c_id = str(c)
            c_name = '[unknown]'
        print(f"    - {c_id} ({c_name})")
    print()

## 3. Data Extraction
Load data from selected record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# List the available record set @id values again, for clarity
print('Available record set @id values:')
pprint(record_set_ids)

# Choose a record set to load (for demonstration, pick the first one if available)
chosen_record_set_id = None
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]  # Replace with a specific @id as needed
    print(f'Using record set for analysis: {chosen_record_set_id}')
else:
    print('No record sets found in the metadata. Exiting data extraction.')

# Load data from all record sets found
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for {record_set_id}: {df.shape[0]} rows × {df.shape[1]} columns.")
        print("Columns (by field @id):", df.columns.tolist())
        display(df.head(3))
    else:
        print(f"Warning: No records found for {record_set_id}.")

# For demonstration, use the first available DataFrame
df = None
if chosen_record_set_id and chosen_record_set_id in dataframes:
    df = dataframes[chosen_record_set_id]

# Show its top rows
if df is not None:
    print(f"First rows from DataFrame for {chosen_record_set_id}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply standard processing: filtering, normalization, and groupby operations referencing fields by their `@id`.

In [ ]:
import numpy as np

if df is not None:
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric fields available (by @id):', numeric_fields)
    # Pick the first numeric field for demonstration
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 1
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the chosen numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical/string field (by @id)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by {group_field_id} and aggregating {numeric_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print('No suitable non-numeric column found for grouping.')
    else:
        print('No numeric fields found in this DataFrame.')
else:
    print('No DataFrame available for EDA.')

## 5. Visualization
Visualize distributions or relationships of the most important fields in the dataset. All references should use `@id` of columns/fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If we have a categorical field for groupby, show a boxplot
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=numeric_field_id, x=group_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print('No numeric fields available for plotting.')

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and perform basic analysis on the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

- All extractions and analyses referenced dataset entities by their `@id` values as required for FAIR reproducibility.
- We loaded dataset metadata, reviewed record set and field structure, extracted records, filtered and normalized data, and generated insightful visualizations.

The same workflow can be extended to more advanced analysis and integration with other Croissant-compatible datasets.